# Verify, demonstrate, and preliminarily analyze M4

M4 validates clean source/test change dynamics for RQ2 under the current artifact policy. It does not measure coordination friction or causality.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import plotly.express as px
from IPython.display import Image, display
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'paper_v9').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
METRICS = ROOT / 'paper_v9' / 'data' / 'metrics'
FIGURES = ROOT / 'paper_v9' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
magnitude = pd.read_csv(METRICS / 'm4_churn_magnitude.csv', dtype={'Semestre': str})
intensity = pd.read_csv(METRICS / 'm4_commit_intensity.csv', dtype={'Semestre': str})
composition = pd.read_csv(METRICS / 'm4_artifact_composition.csv', dtype={'Semestre': str})
rolling = pd.read_csv(METRICS / 'm4_rolling_7day_trajectory.csv', dtype={'Semestre': str})
pooled_magnitude = pd.read_csv(METRICS / 'm4_churn_magnitude_pooled.csv')
pooled_intensity = pd.read_csv(METRICS / 'm4_commit_intensity_pooled.csv')
pooled_composition = pd.read_csv(METRICS / 'm4_artifact_composition_pooled.csv')
pooled_rolling = pd.read_csv(METRICS / 'm4_rolling_7day_trajectory_pooled.csv')
metadata = json.loads((METRICS / 'm4_clean_change_dynamics.metadata.json').read_text())


In [ ]:
assert len(magnitude) == 42
assert len(intensity) == 42
assert len(rolling) == 994
assert len(pooled_magnitude) == 3
assert len(pooled_intensity) == 3
assert len(pooled_composition) > 0
assert len(pooled_rolling) == 71
assert 'Semestre' not in pooled_magnitude.columns
assert 'Semestre' not in pooled_rolling.columns
assert magnitude['clean_churn'].ge(0).all()
assert intensity['median_clean_churn_per_touching_commit'].dropna().ge(0).all()
assert composition['churn_lines'].ge(0).all()
assert rolling['clean_churn_7d'].ge(0).all()
assert pooled_rolling['total_clean_churn_7d'].ge(0).all()
assert set(composition['file_category']) <= {'clean_source_or_test', 'excluded_or_non_measurement'}
assert metadata['policy_version'] == 'code-churn-metrics-v2'
assert metadata['rq'] == 'RQ2'
print('M4 schemas, pooled grains, policy, ranges, coverage, and RQ linkage: PASS')

In [ ]:
magnitude['team_semester'] = magnitude['Semestre'] + '/' + magnitude['ID_Equipe']
magnitude_figure = px.bar(magnitude, x='team_semester', y='clean_churn', color='temporal_marker', title='M4 clean churn magnitude')
magnitude_figure.update_yaxes(title='Clean source/test churn')
rolling_summary = rolling.groupby(['Semestre','window_end_day'], as_index=False).agg(clean_churn_7d=('clean_churn_7d','sum'))
rolling_figure = px.line(rolling_summary, x='window_end_day', y='clean_churn_7d', color='Semestre', markers=True, title='M4 clean churn rolling trajectory')
rolling_figure.add_vline(x=0, line_dash='dash')
pooled_magnitude_figure = px.bar(pooled_magnitude, x='temporal_marker', y='total_clean_churn', title='M4 clean churn pooled by temporal marker')
pooled_rolling_figure = px.line(pooled_rolling, x='window_end_day', y='total_clean_churn_7d', markers=True, title='M4 clean churn pooled rolling trajectory')
pooled_rolling_figure.add_vline(x=0, line_dash='dash')
for figure, stem in ((magnitude_figure, 'm4_clean_churn_magnitude'), (rolling_figure, 'm4_clean_churn_rolling_7day'), (pooled_magnitude_figure, 'm4_clean_churn_magnitude_pooled'), (pooled_rolling_figure, 'm4_clean_churn_rolling_7day_pooled')):
    figure.write_html(METRICS / f'{stem}.html', include_plotlyjs='cdn')
    for extension in ('pdf','svg','png'): figure.write_image(FIGURES / f'{stem}.{extension}', scale=2 if extension == 'png' else 1)
for stem in ('m4_clean_churn_magnitude', 'm4_clean_churn_rolling_7day', 'm4_clean_churn_magnitude_pooled', 'm4_clean_churn_rolling_7day_pooled'):
    assert all((FIGURES / f'{stem}.{extension}').is_file() for extension in ('pdf','svg','png'))
print('M4 stratified and semester-independent article-ready figures generated: PASS')

In [ ]:
display(magnitude[['Semestre','ID_Equipe','temporal_marker','clean_churn','clean_unique_file_n','clean_share_of_all_churn']].head(14))
display(composition.groupby(['file_category','included_in_m4'], as_index=False).agg(churn_lines=('churn_lines','sum'), file_event_n=('file_event_n','sum')))
display(pooled_magnitude)
display(pooled_rolling[pooled_rolling['window_end_day'].isin([-7, 0, 7])])
for path in (FIGURES / 'm4_clean_churn_magnitude.png', FIGURES / 'm4_clean_churn_rolling_7day.png', FIGURES / 'm4_clean_churn_magnitude_pooled.png', FIGURES / 'm4_clean_churn_rolling_7day_pooled.png'):
    assert path.is_file() and path.stat().st_size > 0
    display(Image(filename=str(path), width=900))
print('M4 artifact demo: stratified and pooled tables and figures displayed')

## Preliminary RQ2 reading

M4 quantifies how much source/test change is observed and how clean-change magnitude evolves around the T3 anchor. It complements M3 timing/authorship context and should be contrasted descriptively with M5 coordination evidence. It does not identify friction, effort, productivity, quality, or causality.

In [ ]:
summary = pd.DataFrame([{
    'team_semesters_checkpoints': len(magnitude),
    'clean_churn_total_stratified': magnitude['clean_churn'].sum(),
    'clean_churn_median_stratified': magnitude['clean_churn'].median(),
    'pooled_temporal_markers': len(pooled_magnitude),
    'pooled_clean_churn_total': pooled_magnitude['total_clean_churn'].sum(),
    'pooled_rolling_rows': len(pooled_rolling),
    'rolling_rows_stratified': len(rolling),
}])
display(summary)
print('preliminary RQ2 reading: pooled M4 clean-change summaries reveal the common temporal shape across semesters; stratified outputs remain necessary for cohort differences and gaps')